In [1]:
# STEP2_feature_engineering.py

import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# HELPER FUNCTIONS (No pandas_ta / ta lib!)
# ─────────────────────────────────────────

def compute_rsi(close, period=14):
    """
    RSI = Relative Strength Index (0-100)
    Above 70 = overbought (possible SELL)
    Below 30 = oversold (possible BUY)
    """
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def compute_macd(close, fast=12, slow=26, signal=9):
    """
    MACD = Moving Average Convergence Divergence
    Shows trend direction and momentum.
    Returns: MACD line, Signal line, Histogram
    """
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    histogram = macd - signal_line
    return macd, signal_line, histogram


def compute_bollinger_bands(close, period=20):
    """
    Bollinger Bands = Price bands around a moving average.
    Width tells us how volatile the stock is.
    """
    ma = close.rolling(window=period).mean()
    std = close.rolling(window=period).std()
    upper = ma + 2 * std
    lower = ma - 2 * std
    width = (upper - lower) / ma * 100  # as percentage
    return upper, lower, width


def compute_atr(high, low, close, period=14):
    """
    ATR = Average True Range
    Measures how much the price moves on average each day.
    Higher ATR = more volatile.
    """
    hl = high - low
    hc = (high - close.shift()).abs()
    lc = (low - close.shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.rolling(window=period).mean()


def compute_obv(close, volume):
    """
    OBV = On Balance Volume
    Tracks whether volume is flowing into or out of a stock.
    """
    direction = np.sign(close.diff())
    direction.iloc[0] = 0
    return (direction * volume).cumsum()


def compute_stochastic(high, low, close, k_period=14, d_period=3):
    """
    Stochastic Oscillator
    Compares closing price to recent high-low range.
    %K > 80 = overbought, %K < 20 = oversold
    """
    lowest_low = low.rolling(window=k_period).min()
    highest_high = high.rolling(window=k_period).max()
    k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    d = k.rolling(window=d_period).mean()
    return k, d


def compute_cci(high, low, close, period=20):
    """
    CCI = Commodity Channel Index
    Measures how far price is from its average.
    Above +100 = strong uptrend, Below -100 = strong downtrend
    """
    typical_price = (high + low + close) / 3
    ma = typical_price.rolling(window=period).mean()
    md = typical_price.rolling(window=period).apply(
        lambda x: np.mean(np.abs(x - x.mean())), raw=True
    )
    return (typical_price - ma) / (0.015 * md)


def compute_williams_r(high, low, close, period=14):
    """
    Williams %R
    Similar to stochastic; measures overbought/oversold.
    Range: -100 to 0. Above -20 = overbought, Below -80 = oversold.
    """
    highest_high = high.rolling(window=period).max()
    lowest_low = low.rolling(window=period).min()
    return -100 * (highest_high - close) / (highest_high - lowest_low)


def compute_ema(close, span):
    """Exponential Moving Average"""
    return close.ewm(span=span, adjust=False).mean()


def compute_roc(close, period=10):
    """
    ROC = Rate of Change
    How much the price changed over N days (in %)
    """
    return (close - close.shift(period)) / close.shift(period) * 100


def compute_mfi(high, low, close, volume, period=14):
    """
    MFI = Money Flow Index
    Like RSI but includes volume. Shows buying/selling pressure.
    """
    typical_price = (high + low + close) / 3
    money_flow = typical_price * volume
    direction = typical_price.diff()
    pos_flow = money_flow.where(direction > 0, 0)
    neg_flow = money_flow.where(direction < 0, 0)
    pos_mf = pos_flow.rolling(window=period).sum()
    neg_mf = neg_flow.rolling(window=period).sum()
    mfr = pos_mf / neg_mf
    return 100 - (100 / (1 + mfr))


# ─────────────────────────────────────────
# MAIN FEATURE ENGINEERING FUNCTION
# ─────────────────────────────────────────

def engineer_features(df):
    """
    Takes raw OHLCV data for ONE stock and returns
    a DataFrame with 19+ technical indicator features.
    """
    df = df.copy().sort_values("Date").reset_index(drop=True)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]

    # --- Moving Averages ---
    df["MA_5"]  = close.rolling(5).mean()   # 5-day average
    df["MA_10"] = close.rolling(10).mean()  # 10-day average
    df["MA_20"] = close.rolling(20).mean()  # 20-day average

    # --- EMA ---
    df["EMA_12"] = compute_ema(close, 12)
    df["EMA_26"] = compute_ema(close, 26)

    # --- RSI ---
    df["RSI"] = compute_rsi(close, 14)

    # --- MACD ---
    df["MACD"], df["MACD_Signal"], df["MACD_Hist"] = compute_macd(close)

    # --- Bollinger Bands ---
    df["BB_Upper"], df["BB_Lower"], df["BB_Width"] = compute_bollinger_bands(close)

    # --- ATR ---
    df["ATR"] = compute_atr(high, low, close, 14)

    # --- OBV ---
    df["OBV"] = compute_obv(close, vol)

    # --- Stochastic Oscillator ---
    df["Stoch_K"], df["Stoch_D"] = compute_stochastic(high, low, close)

    # --- CCI ---
    df["CCI"] = compute_cci(high, low, close)

    # --- Williams %R ---
    df["Williams_R"] = compute_williams_r(high, low, close)

    # --- Rate of Change ---
    df["ROC_10"] = compute_roc(close, 10)

    # --- Money Flow Index ---
    df["MFI"] = compute_mfi(high, low, close, vol)

    # --- Volume MA ratio (volume vs its own average) ---
    df["Vol_Ratio"] = vol / vol.rolling(20).mean()

    return df


# ─────────────────────────────────────────
# RUN IT ON ALL STOCKS
# ─────────────────────────────────────────

raw = pd.read_csv("indian_realty_clean_data.csv", parse_dates=["Date"])

all_featured = []

for ticker, group in raw.groupby("Ticker"):
    print(f"Computing features for {ticker}...")
    featured = engineer_features(group)
    all_featured.append(featured)

df_featured = pd.concat(all_featured, ignore_index=True)

# Drop rows where indicators couldn't be calculated (NaN at start)
df_featured.dropna(inplace=True)
df_featured.reset_index(drop=True, inplace=True)

df_featured.to_csv("indian_realty_features.csv", index=False)
print(f"\n✅ Features computed! Shape: {df_featured.shape}")
print(f"📊 Columns: {list(df_featured.columns)}")
print(df_featured.head(3))

Computing features for ANANTRAJ.NS...
Computing features for ARVSMART.NS...
Computing features for BRIGADE.NS...
Computing features for DBREALTY.NS...
Computing features for DLF.NS...
Computing features for GODREJPROP.NS...
Computing features for IBREALEST.NS...
Computing features for KOLTEPATIL.NS...
Computing features for LODHA.NS...
Computing features for MAHLIFE.NS...
Computing features for OBEROIRLTY.NS...
Computing features for PENINLAND.NS...
Computing features for PHOENIXLTD.NS...
Computing features for PRESTIGE.NS...
Computing features for RUSTOMJEE.NS...
Computing features for SHRIRAMPPS.NS...
Computing features for SIGNATURE.NS...
Computing features for SOBHA.NS...
Computing features for SUNTECK.NS...

✅ Features computed! Shape: (23456, 28)
📊 Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'MA_5', 'MA_10', 'MA_20', 'EMA_12', 'EMA_26', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Width', 'ATR', 'OBV', 'Stoch_K', 'Stoch_D', 'CCI